# LeetCode #79: Word Search

https://leetcode.com/problems/word-search/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(m \cdot n \cdot 4^L)$ | $O(m \cdot n)$ |
| **Optimal: DFS + In-place Backtracking ★** | $O(m \cdot n \cdot 4^L)$ | $O(L)$ |

---

## Understanding the Methods

### Brute Force
Maintain a separate `visited` boolean matrix of size m×n. For every DFS call, copy or update the matrix and restore it afterward. Asymptotically equivalent but allocates $O(m \cdot n)$ extra memory for the visited grid.

### Optimal: DFS + In-place Backtracking ★
Mark a visited cell by temporarily overwriting it with a sentinel character ('#') that cannot appear in the word. After the recursive call returns, restore the original character. This eliminates the separate visited matrix entirely.

**Why this is better than Brute Force:** In-place marking keeps space at $O(L)$ (call stack depth equals word length) instead of $O(m \cdot n)$ for the visited grid. All other work is identical.

**Constraints:**
* m == board.length, n == board[i].length
* 1 <= m, n <= 6
* 1 <= word.length <= 15
* board and word consist of only lowercase and uppercase English letters

## Solutions

### C#

In [ ]:
public class Solution {
    public bool Exist(char[][] board, string word) {
        int m = board.Length, n = board[0].Length;
        for (int r = 0; r < m; r++)
            for (int c = 0; c < n; c++)
                if (Dfs(board, word, r, c, 0)) return true;
        return false;
    }

    private bool Dfs(char[][] board, string word, int r, int c, int idx) {
        // All characters matched — word found
        if (idx == word.Length) return true;
        if (r < 0 || r >= board.Length || c < 0 || c >= board[0].Length) return false;
        // Cell was already visited on this path, or letter doesn't match
        if (board[r][c] != word[idx]) return false;

        // Mark this cell visited by overwriting it with a sentinel
        char tmp = board[r][c];
        board[r][c] = '#';
        bool found = Dfs(board, word, r + 1, c, idx + 1)
                  || Dfs(board, word, r - 1, c, idx + 1)
                  || Dfs(board, word, r, c + 1, idx + 1)
                  || Dfs(board, word, r, c - 1, idx + 1);
        // Restore the original letter so other search paths can use this cell
        board[r][c] = tmp;
        return found;
    }
}

### Python

In [ ]:
class Solution:
    def exist(self, board: list[list[str]], word: str) -> bool:
        m, n = len(board), len(board[0])

        def dfs(r: int, c: int, idx: int) -> bool:
            # All characters matched — word found
            if idx == len(word):
                return True
            if r < 0 or r >= m or c < 0 or c >= n:
                return False
            # Cell was already visited on this path, or letter doesn't match
            if board[r][c] != word[idx]:
                return False

            # Mark this cell visited by overwriting it with a sentinel
            tmp, board[r][c] = board[r][c], '#'
            found = (dfs(r + 1, c, idx + 1) or dfs(r - 1, c, idx + 1)
                     or dfs(r, c + 1, idx + 1) or dfs(r, c - 1, idx + 1))
            # Restore the original letter so other search paths can use this cell
            board[r][c] = tmp
            return found

        for r in range(m):
            for c in range(n):
                if dfs(r, c, 0):
                    return True
        return False

### Go

In [ ]:
func exist(board [][]byte, word string) bool {
    m, n := len(board), len(board[0])

    var dfs func(r, c, idx int) bool
    dfs = func(r, c, idx int) bool {
        // All characters matched — word found
        if idx == len(word) {
            return true
        }
        if r < 0 || r >= m || c < 0 || c >= n {
            return false
        }
        // Cell was already visited on this path, or letter doesn't match
        if board[r][c] != word[idx] {
            return false
        }
        // Mark this cell visited by overwriting it with a sentinel
        tmp := board[r][c]
        board[r][c] = '#'
        found := dfs(r+1, c, idx+1) || dfs(r-1, c, idx+1) ||
            dfs(r, c+1, idx+1) || dfs(r, c-1, idx+1)
        // Restore the original letter so other search paths can use this cell
        board[r][c] = tmp
        return found
    }

    for r := 0; r < m; r++ {
        for c := 0; c < n; c++ {
            if dfs(r, c, 0) {
                return true
            }
        }
    }
    return false
}

### Rust

In [ ]:
impl Solution {
    pub fn exist(mut board: Vec<Vec<char>>, word: String) -> bool {
        let word: Vec<char> = word.chars().collect();
        let m = board.len();
        let n = board[0].len();
        for r in 0..m {
            for c in 0..n {
                if Self::dfs(&mut board, &word, r, c, 0) {
                    return true;
                }
            }
        }
        false
    }

    fn dfs(board: &mut Vec<Vec<char>>, word: &[char], r: usize, c: usize, idx: usize) -> bool {
        // All characters matched — word found
        if idx == word.len() { return true; }
        if board[r][c] != word[idx] { return false; }
        // Mark this cell visited by overwriting it with a sentinel
        let tmp = board[r][c];
        board[r][c] = '#';
        let dirs: [(i32, i32); 4] = [(1,0),(-1,0),(0,1),(0,-1)];
        let found = dirs.iter().any(|&(dr, dc)| {
            let nr = r as i32 + dr;
            let nc = c as i32 + dc;
            if nr >= 0 && nr < board.len() as i32 && nc >= 0 && nc < board[0].len() as i32 {
                Self::dfs(board, word, nr as usize, nc as usize, idx + 1)
            } else { false }
        });
        // Restore the original letter so other search paths can use this cell
        board[r][c] = tmp;
        found
    }
}

## Example Scenarios

**1. Common Case**
**Input:** board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], word = "ABCCED"
Starting at (0,0)='A': → (0,1)='B' → (0,2)='C' → (1,2)='C' → (2,2)='E' → (2,1)='D'. All 6 characters match; the sentinel '#' prevents revisiting (0,2) when we pass through (1,2). Return true.

**2. Slightly Complex**
**Input:** board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], word = "SEE"
Multiple starting 'S' cells exist. The DFS starting at (1,3)='S' reaches (2,3)='E' → (2,2)='E' and matches all three characters. The sentinel correctly blocks the path from revisiting (2,3) when exploring from it.

**3. Edge Case: Time Factor**
**Input:** A 6×6 board of all 'A's, word = "AAAAAAAAAAAAAAB" (14 A's followed by 'B', length 15)
The DFS explores nearly every path before concluding 'B' is not present. At each step, up to 4 neighbors are tried; the recursion tree approaches $4^{15}$ leaves, but the grid constraint (6×6 = 36 cells) and sentinel marking cap the actual explored paths far below $4^{15}$.

**4. Edge Case: Space Factor**
**Input:** word of length L = 15 on a 6×6 grid.
Stack depth is at most L = 15. Each frame stores only the current (r, c, idx) — three integers. No visited matrix is allocated. Peak auxiliary memory is $O(15) = O(L)$ stack frames, regardless of board size.

**5. Almost-Impossible but Plausible**
**Input:** board = [["A"]], word = "A"
Single-cell board, single-character word. The outer loop starts at (0,0): dfs(0, 0, 0) checks board[0][0]=='A' == word[0], marks '#', recurses with idx=1 which equals len(word)=1 → returns true immediately. The sentinel is restored even though the result is already known.